# Notebook 2: A signal next to its expectation

A signal is a quantity the system expects: with a meaning, a unit, a plausible range and an operating range. This
notebook puts the data of a signal next to that expectation: the raw series and the clean series one over the other,
the values outside the plausible range marked, the operating range as a band. At the end it shows what the decision
of the hydrogen chain says about its inputs today.

The notebook only reads. It needs the retrieval server (path A).

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from opentsx.client import RetrievalClient

RETRIEVAL = os.environ.get("OPENTSX_RETRIEVAL", "http://opentsx-retrieval.opentsx.svc:8080")
SYSTEM = "h2-chain"
client = RetrievalClient(RETRIEVAL)
print("Retrieval server:", RETRIEVAL)

## 1. The signals of the system

One row per signal: role, state, origin, expectation and the series the system found for the grades `raw` and
`clean`.

In [ ]:
signals = client.signals(SYSTEM)
signals[["id", "role", "state", "origin", "unit", "min", "max", "operating_min", "operating_max", "interval", "next"]]

## 2. Choose a signal

The chosen signal is the wind of the DWD station at the park: measured 10 m above ground, present raw and clean.
Other signals with data: `park.p_potential_kw` (estimated from the wind), `market.price_eur_mwh` (reported),
`offtake.demand_kgph` (generated by a demo generator).

In [ ]:
SIGNAL = "park/DWD-01759/wind_speed"
START, END = "2025-10-01T00:00:00Z", "2025-11-01T00:00:00Z"

sig = signals.set_index("id").loc[SIGNAL]
print(f"{SIGNAL}: {sig['meaning']}")
print(f"Expectation: {sig['unit']}, plausible {sig['min']} to {sig['max']}, operating {sig['operating_min']} to {sig['operating_max']}, interval {sig['interval']}")
print(f"State {sig['state']}, origin {sig['origin']}")


def series_of(ids):
    if not isinstance(ids, str) or not ids:
        return pd.Series(dtype=float)
    f = client.frame(series=ids.split(","), start=START, end=END)
    return f.set_index("ts")["value"].sort_index() if len(f) else pd.Series(dtype=float)


raw = series_of(sig["raw_series"])
clean = series_of(sig["clean_series"])
print(f"raw: {len(raw)} values, clean: {len(clean)} values between {START} and {END}")

## 3. Raw and clean, one over the other

The raw series in grey, the clean series in blue. Red dots are raw values outside the plausible range; the cleaning
turns them into gaps and does not invent values for them. The band is the operating range: values outside it are
possible, but they are not normal operation.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
if pd.notna(sig["operating_min"]) and pd.notna(sig["operating_max"]):
    ax.axhspan(sig["operating_min"], sig["operating_max"], color="#e3efe3", label="operating range")
if len(raw):
    raw.plot(ax=ax, lw=0.5, color="#8a949e", label="raw")
if len(clean):
    clean.plot(ax=ax, lw=0.8, color="#1f6fb3", label="clean")
if len(raw) and pd.notna(sig["min"]) and pd.notna(sig["max"]):
    out = raw[(raw < sig["min"]) | (raw > sig["max"])]
    ax.scatter(out.index, out.values, color="#b3392b", s=12, zorder=3, label=f"outside plausible ({len(out)})")
ax.set_ylabel(sig["unit"] or "")
ax.set_title(SIGNAL)
ax.legend(loc="upper right")
plt.show()

## 4. What the cleaning changed

The same times side by side: how many values stayed the same, how many became a gap, and the largest differences.

In [ ]:
if len(raw) and len(clean):
    both = pd.concat({"raw": raw, "clean": clean}, axis=1)
    same = (both["raw"] - both["clean"]).abs() < 1e-9
    print(f"same: {int(same.sum())}, changed into a gap: {int((both['raw'].notna() & both['clean'].isna()).sum())}, "
          f"only clean (other interval): {int((both['raw'].isna() & both['clean'].notna()).sum())}")
    display(both[~same].dropna(how="all").head(10))
else:
    print("This signal does not have both grades in the chosen period.")

## 5. What the decision says about its inputs

The decision card of the hydrogen chain names the signals it needs and what makes them fit for its purpose: grade,
age, allowed origin. From that follows `usable`, `restricted` or `blocked` for each analysis, with the reason.

In [ ]:
status = client.decisions(SYSTEM)
for d in status["decisions"]:
    print(f"{d['title']} (data as of {d.get('asOf', 'n/a')})")
    print(pd.DataFrame([{"signal": n["signal"], "grade": n.get("grade", "clean"), "fit": n["ok"],
                         "reason": "; ".join(n["reasons"])} for n in d["needs"]]).to_string(index=False))
    for a in d["analyses"]:
        print(f"\n  {a['title']}: {a['status']}")
        for r in a["reasons"]:
            print(f"    - {r}")

The status here reads the system document without the working copy of an account. What was bound in the web
application is shown on the place "Models".

Sources: Deutscher Wetterdienst, CC BY 4.0; Bundesnetzagentur | SMARD.de, CC BY 4.0.